In [1]:
# Import the libraries we need
import pandas as pd
import numpy as np

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
# Load the two raw CSV files
vehicle_df = pd.read_csv("dft-road-casualty-statistics-vehicle-2024.csv")
collision_df = pd.read_csv("dft-road-casualty-statistics-collision-2024.csv")

# Check the number of rows and columns
print("Vehicle data:", vehicle_df.shape)
print("Collision data:", collision_df.shape)

C:\Users\Hp\AppData\Local\Temp\ipykernel_6284\2123514613.py:2: DtypeWarning: Columns (27: generic_make_model) have mixed types. Specify dtype option on import or set low_memory=False.
  vehicle_df = pd.read_csv("dft-road-casualty-statistics-vehicle-2024.csv")


Vehicle data: (183514, 32)
Collision data: (100927, 44)


C:\Users\Hp\AppData\Local\Temp\ipykernel_6284\2123514613.py:3: DtypeWarning: Columns (0: collision_index, 2: collision_ref_no, 39: lsoa_of_accident_location) have mixed types. Specify dtype option on import or set low_memory=False.
  collision_df = pd.read_csv("dft-road-casualty-statistics-collision-2024.csv")


## Clean the Collision Data

**Why we drop these columns:**
1. **Leakage:** `enhanced_severity_collision`, `collision_injury_based`, `collision_adjusted_severity_*` are all derived from our target. Using them would be cheating.
2. **Historic:** The `*_historic` columns are old versions replaced by the 2024 spec.
3. **Useless / High Cardinality:** Constant columns, exact location coordinates, and `police_force` (too many categories).

**Time features:**
We convert the text `time` and `date` into `hour`, `is_peak`, `month`, and `is_weekend`.

In [4]:
# 1. Create a list of columns to drop
collision_drop = [
    # Leakage (derived from the target)
    "enhanced_severity_collision", "collision_injury_based",
    "collision_adjusted_severity_serious", "collision_adjusted_severity_slight",
    
    # Historic versions (replaced in 2024)
    "junction_detail_historic", "pedestrian_crossing_human_control_historic",
    "pedestrian_crossing_physical_facilities_historic", "carriageway_hazards_historic",
    
    # Constant or useless
    "collision_year", "local_authority_district",
    
    # High cardinality (coords, LSOA, police force)
    "lsoa_of_accident_location", "location_easting_osgr", "location_northing_osgr",
    "longitude", "latitude", "local_authority_ons_district", 
    "local_authority_highway", "local_authority_highway_current", "police_force"
]

# 2. Drop the columns (we use .copy() so pandas doesn't complain)
collision_clean = collision_df.drop(
    columns=[c for c in collision_drop if c in collision_df.columns]
).copy()

# 3. Extract hour and peak travel times from the 'time' column
collision_clean["hour"] = pd.to_datetime(
    collision_clean["time"], format="%H:%M", errors="coerce"
).dt.hour

# is_peak = 1 if hour is 7,8,9 (morning) or 16,17,18 (evening)
collision_clean["is_peak"] = collision_clean["hour"].isin([7, 8, 9, 16, 17, 18]).astype(int)

# 4. Extract month and weekend flag from the 'date' column
collision_clean["date"] = pd.to_datetime(
    collision_clean["date"], format="%d/%m/%Y", errors="coerce"
)
collision_clean["month"] = collision_clean["date"].dt.month

# day_of_week: 1 = Sunday, 7 = Saturday
collision_clean["is_weekend"] = collision_clean["day_of_week"].isin([1, 7]).astype(int)

# 5. Drop the original text columns since we extracted what we need
collision_clean = collision_clean.drop(columns=["time", "date"])

print("Collision data cleaned. New shape:", collision_clean.shape)

Collision data cleaned. New shape: (100927, 27)


## Replace missing/unknown codes with NaN

According to the data guide:
- `-1` means "Data missing or out of range"
- `9` and `99` mean "unknown (self reported)"

If we leave these as numbers, our model will think a speed limit of `-1` is a real speed limit. 
We will replace them with `NaN` (blank), so our teammates know they are missing.

In [5]:
# List of columns that use numerical codes for missing/unknown data
code_cols = [
    "speed_limit", "road_type", "junction_detail", "junction_control",
    "pedestrian_crossing", "light_conditions", "weather_conditions",
    "road_surface_conditions", "special_conditions_at_site",
    "carriageway_hazards", "first_road_class", "second_road_class",
    "trunk_road_flag", "did_police_officer_attend_scene_of_accident",
]

# Replace -1, 9, and 99 with NaN in these columns
for col in code_cols:
    if col in collision_clean.columns:
        collision_clean[col] = collision_clean[col].replace([-1, 9, 99], np.nan)

# Special case: Urban/Rural uses 3 for "Unallocated" and -1 for missing
collision_clean["urban_or_rural_area"] = (
    collision_clean["urban_or_rural_area"].replace([3, -1], np.nan)
)

# Check how many missing values we have now
print("Missing values in key columns:")
print(collision_clean[["speed_limit", "weather_conditions", "road_surface_conditions"]].isna().sum())

Missing values in key columns:
speed_limit                   3
weather_conditions         3145
road_surface_conditions    2374
dtype: int64


## Clean the Vehicle Data

Before we group the vehicles by collision, we need to:
1. Drop the columns we don't need (historic versions and high-cardinality text).
2. Replace the missing/unknown codes (`-1`, `9`, `99`, `999`) with `NaN`.

In [6]:
# 1. Drop unnecessary vehicle columns
vehicle_drop = [
    "collision_year", "vehicle_manoeuvre_historic", 
    "vehicle_location_restricted_lane_historic", "journey_purpose_of_driver_historic",
    "lsoa_of_driver", "generic_make_model",
]

vehicle_clean = vehicle_df.drop(
    columns=[c for c in vehicle_drop if c in vehicle_df.columns]
).copy()

# 2. Replace missing codes in all numeric columns
for col in vehicle_clean.select_dtypes(include="number").columns:
    vehicle_clean[col] = vehicle_clean[col].replace([-1, 9, 99, 999], np.nan)

print("Vehicle data cleaned. Shape:", vehicle_clean.shape)

Vehicle data cleaned. Shape: (183514, 26)


## Collapse vehicle data to one row per collision

A single collision can involve multiple vehicles. If we merge the vehicle file as-is, we will duplicate the crash severity for every vehicle involved. 

To fix this, we group the data by `collision_index` and create summary features:
- `num_vehicles_recorded`: How many vehicles were in the crash.
- `has_motorcycle`, `has_pedal_cycle`, `has_hgv`, `has_bus`: Yes/No flags for vehicle types.
- `max_driver_age` & `min_driver_age`: The oldest and youngest drivers involved.
- `any_skidding`: 1 if any vehicle skidded, 0 otherwise.
- `num_left_hand_drive`: Count of left-hand-drive vehicles.
- `any_escooter`: 1 if an e-scooter was involved.

In [7]:
# Group by collision_index and create summary features
vehicle_summary = (
    vehicle_clean
    .groupby("collision_index")
    .agg(
        num_vehicles_recorded = ("vehicle_reference", "count"),
        
        # Vehicle types (based on codes from the data guide)
        has_motorcycle        = ("vehicle_type", lambda x: int(x.isin([2,3,4,5,23,97]).any())),
        has_pedal_cycle       = ("vehicle_type", lambda x: int((x == 1).any())),
        has_hgv               = ("vehicle_type", lambda x: int(x.isin([19,20,21,98]).any())),
        has_bus               = ("vehicle_type", lambda x: int(x.isin([10,11]).any())),
        
        # Driver ages
        max_driver_age        = ("age_of_driver", "max"),
        min_driver_age        = ("age_of_driver", "min"),
        
        # Other features
        any_skidding          = ("skidding_and_overturning", lambda x: int((x.fillna(0) != 0).any())),
        num_left_hand_drive   = ("vehicle_left_hand_drive", lambda x: int((x == 1).sum())),
        any_escooter          = ("escooter_flag", lambda x: int((x == 1).any())),
    )
    .reset_index()
)

print("Vehicle summary shape:", vehicle_summary.shape)

Vehicle summary shape: (100927, 11)


## Merge the datasets

We use a **left join** to keep all 100,927 collisions from our main collision dataset, and attach the vehicle summary columns where they match on `collision_index`.

We then verify that the row count stayed exactly at 100,927 and that our target variable (`collision_severity`) is intact.

In [8]:
# 1. Merge the data
merged = collision_clean.merge(vehicle_summary, on="collision_index", how="left")

# 2. Print the shape and severity counts to verify
print("Merged shape:", merged.shape)
print("\nCollision Severity Distribution:")
print(merged["collision_severity"].value_counts().sort_index())

Merged shape: (100927, 37)

Collision Severity Distribution:
collision_severity
1     1502
2    23567
3    75858
Name: count, dtype: int64


## Save the clean merged file

Finally, we save the merged dataset to a new CSV file.

In [9]:
# Save the merged dataframe to a CSV file
merged.to_csv("collisions_merged_clean.csv", index=False)

print("✅ Saved successfully as collisions_merged_clean.csv")

✅ Saved successfully as collisions_merged_clean.csv
